# Scheduled hosted-layer refresh
> Runs as an **ArcGIS Online Hosted Notebook** (Standard runtime). `GIS('home')` uses the notebook's own session — no credentials in code. Logic lives in the `agolops` package (tested in CI); cells only orchestrate.

Replaces the contents of a hosted feature layer from an upstream source, with validation and a shrink guard **before** anything touches production. Schedule nightly via the **Tasks** pane.

In [ ]:
# Install the tested logic (a hosted notebook starts from a clean runtime, without git)
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "https://github.com/iDan-Armstrong/agol-ops-notebooks/archive/refs/heads/main.zip",
])

In [ ]:
from arcgis.gis import GIS

from agolops.refresh import plan_refresh, validate_against_layer

gis = GIS("home")

TARGET_ITEM_ID = "REPLACE_WITH_ITEM_ID"   # hosted feature layer to refresh
SOURCE_ITEM_ID = "REPLACE_WITH_ITEM_ID"   # upstream CSV/GeoJSON item (or swap in an API call)
REQUIRED_FIELDS = ["facility_name"]
KEY_FIELD = "facility_id"

In [ ]:
# Fetch new rows from the source item
import json

source = gis.content.get(SOURCE_ITEM_ID)
rows = json.loads(source.get_data(try_json=False).decode())  # expects a list of dicts
print(f"source rows: {len(rows)}")

In [ ]:
# Validate against the live layer, then decide
target = gis.content.get(TARGET_ITEM_ID)
layer = target.layers[0]
layer_fields = [f.name for f in layer.properties.fields]
current_count = layer.query(return_count_only=True)

validation = validate_against_layer(rows, layer_fields, REQUIRED_FIELDS, KEY_FIELD)
plan = plan_refresh(len(rows), current_count, validation)
print(plan)
if plan.action == "abort":
    raise SystemExit(plan.reason)  # scheduled run shows as Failed, nobody loses data

In [ ]:
# Execute: truncate + chunked append (keeps the layer, its item id, and all its references)
layer.manager.truncate()
features = [{"attributes": row} for row in rows]
for start in range(0, len(features), plan.chunk_size):
    chunk = features[start : start + plan.chunk_size]
    result = layer.edit_features(adds=chunk)
    failed = [r for r in result["addResults"] if not r["success"]]
    if failed:
        raise RuntimeError(f"append failed at offset {start}: {failed[:3]}")
print(f"replaced {current_count} -> {len(features)} features in {plan.chunks} chunks")

Failure modes are explicit: validation errors or a >20% shrink abort before the truncate, and a failed chunk raises immediately. The truncate+append pattern (vs. overwrite) preserves the item id, so maps, apps, and popups keep working.